In [ ]:
import pandas as pd
import os

print("Current working directory:", os.getcwd())
print("Files in project folder:")
for file in os.listdir('.'):
    print(file)

100%|██████████| 20.7M/20.7M [00:00<00:00, 165MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/dhrubangtalukdar/store-item-demand-forecasting-dataset/versions/2
retail_sales.csv


In [ ]:
df = pd.read_csv("retail_sales.csv")
print(df.head())

       date store_id item_id  sales  price  promo  weekday  month
0  1/1/2019  store_1  item_1     41  21.30      0        1      1
1  1/2/2019  store_1  item_1     53  21.30      0        2      1
2  1/3/2019  store_1  item_1     39  21.30      0        3      1
3  1/4/2019  store_1  item_1     35  21.30      0        4      1
4  1/5/2019  store_1  item_1     51  17.04      1        5      1


In [ ]:
import numpy as np

# Ensure the dataset is sorted by date for a time-series workflow.
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

df['year'] = df['date'].dt.year
df['month_num'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['weekday'] = df['date'].dt.weekday

df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

df['store_id'] = df['store_id'].astype('category')
df['item_id'] = df['item_id'].astype('category')

df['lag_1'] = df.groupby(['store_id', 'item_id'])['sales'].shift(1)
df['lag_7'] = df.groupby(['store_id', 'item_id'])['sales'].shift(7)
df['lag_14'] = df.groupby(['store_id', 'item_id'])['sales'].shift(14)
df['lag_30'] = df.groupby(['store_id', 'item_id'])['sales'].shift(30)

df['rolling_mean_7'] = (
    df.groupby(['store_id', 'item_id'])['sales']
      .shift(1)
      .rolling(7)
      .mean()
)
df['rolling_mean_14'] = (
    df.groupby(['store_id', 'item_id'])['sales']
      .shift(1)
      .rolling(14)
      .mean()
)
df['rolling_mean_30'] = (
    df.groupby(['store_id', 'item_id'])['sales']
      .shift(1)
      .rolling(30)
      .mean()
)
df['rolling_std_14'] = (
    df.groupby(['store_id', 'item_id'])['sales']
      .shift(1)
      .rolling(14)
      .std()
)

df = df.dropna().reset_index(drop=True)

print("Dataset after feature engineering:", df.shape)
print(df.dtypes)

/tmp/ipykernel_5929/2896900093.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['lag_1'] = df.groupby(['store_id', 'item_id'])['sales'].shift(1)
/tmp/ipykernel_5929/2896900093.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['lag_7'] = df.groupby(['store_id', 'item_id'])['sales'].shift(7)
/tmp/ipykernel_5929/2896900093.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['store_id', 'item_id'])[

In [ ]:
features = [
    'store_id', 'item_id', 'price', 'promo', 'weekday',
    'month_num', 'year', 'day', 'weekofyear', 'is_weekend',
    'month_sin', 'month_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_14'
]
target = 'sales'

X = df[features]
y = df[target]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [7]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_validate

categorical_features = ['store_id', 'item_id']
numeric_features = [f for f in features if f not in categorical_features]

preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features),
    ('num', StandardScaler(), numeric_features),
])

model_defs = {
    'XGB': Pipeline([
        ('preprocess', preprocessor),
        ('model', XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbosity=0
        ))
    ]),
    'RandomForest': Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=200,
            max_depth=12,
            n_jobs=-1,
            random_state=42
        ))
    ]),
    'Ridge': Pipeline([
        ('preprocess', preprocessor),
        ('model', Ridge(alpha=1.0))
    ])
}

cv = TimeSeriesSplit(n_splits=5)
results = []

for name, pipeline in model_defs.items():
    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=['neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'],
        n_jobs=-1,
        return_train_score=False,
    )
    results.append({
        'model': name,
        'mae': -cv_scores['test_neg_mean_absolute_error'].mean(),
        'rmse': -cv_scores['test_neg_root_mean_squared_error'].mean(),
        'r2': cv_scores['test_r2'].mean(),
    })

scores_df = pd.DataFrame(results).sort_values('mae')
print(scores_df)

best_model_name = scores_df.iloc[0]['model']
best_model = model_defs[best_model_name]
print(f"Best model: {best_model_name}")
best_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("Test MAE :", mae)
print("Test RMSE:", rmse)
print("Test R2  :", r2)

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:300], label='Actual')
plt.plot(pred[:300], label='Predicted')
plt.legend()
plt.title(f"Demand Forecasting - {best_model_name}")
plt.grid()
plt.show()

MAE : 3.380842924118042
RMSE: 4.26276298897451
R2  : 0.9253649711608887


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False)

print(importance_df.head(12))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].head(12)[::-1], importance_df['importance'].head(12)[::-1])
plt.title('Permutation Feature Importance')
plt.xlabel('Mean importance (MAE impact)')
plt.grid(axis='x')
plt.show()

In [ ]:
import joblib

joblib.dump(best_model, "demand_pipeline.pkl")
print("Saved best model pipeline to demand_pipeline.pkl")

['demand_model.pkl']

In [12]:
def generate_insight(prediction):
    if prediction > 170:
        return (
            "High demand expected. Increase inventory and prepare replenishment. "
            "Consider promotional pricing for top items."
        )
    elif prediction > 100:
        return (
            "Moderate demand expected. Maintain stock levels and review promo plans."
        )
    else:
        return (
            "Low demand expected. Avoid overstocking and focus on clearance or bundling." 
        )

sample_prediction = pred[0]
print("Sample prediction:", sample_prediction)
print(generate_insight(sample_prediction))

future_data = pd.DataFrame({
    'store_id': ['store_1'],
    'item_id': ['item_5'],
    'price': [120],
    'promo': [1],
    'weekday': [2],
    'month_num': [6],
    'year': [2026],
    'day': [15],
    'weekofyear': [24],
    'is_weekend': [0],
    'month_sin': [np.sin(2 * np.pi * 6 / 12)],
    'month_cos': [np.cos(2 * np.pi * 6 / 12)],
    'lag_1': [140],
    'lag_7': [135],
    'lag_14': [130],
    'lag_30': [125],
    'rolling_mean_7': [138],
    'rolling_mean_14': [136],
    'rolling_mean_30': [132],
    'rolling_std_14': [4.5],
})

future_prediction = best_model.predict(future_data)
print("Predicted Sales:", future_prediction[0])
print(generate_insight(future_prediction[0]))

Predicted Sales: 90.42797


In [ ]:
# The notebook now includes a more complete forecasting workflow:
# - time-based feature engineering
# - cross-validated model comparison
# - permutation feature importance
# - saved model pipeline
# - future sales prediction and business insights